# Download flat files
This was only possible from April 2024. We simply download everything and extract them to single ticker files.

In [4]:
import os
import pytz
import gzip
import calendar
import pandas as pd
from datetime import datetime, date, time, timedelta
from pytz import timezone
from times import get_market_dates
from fastparquet import write
import boto3
from botocore.config import Config
from dotenv import load_dotenv 

# Now load variables and define constants
load_dotenv() # to load variables from .env

ACCESS_KEY_ID = os.environ.get("ACCESS_KEY_ID")
if not ACCESS_KEY_ID:
    raise ValueError("No API key found. Please set ACCESS_KEY_ID in your .env file.")

API_KEY = os.environ.get("MASSIVE_API_KEY")
if not API_KEY:
    raise ValueError("No API key found. Please set MASSIVE_API_KEY in your .env file.")

session = boto3.Session(
   aws_access_key_id=ACCESS_KEY_ID,
   aws_secret_access_key=API_KEY,
)
s3 = session.client(
   's3',
   endpoint_url='https://files.polygon.io',
   config=Config(signature_version='s3v4'),
)

DATA_PATH = "../data/polygon/"

# START_DATE = date(1999, 9, 10)
# END_DATE = date(2199, 4, 19)

START_DATE = date(2016, 6, 1) # only go as far as your polygon membership allows. don't waste loop cycles
END_DATE = date(2026, 6, 15) # till today only or else you will get "out of range" from our custom function and don't waste loop cycles
# END_DATE = this will need updating everyday so that we only fetch the new adjustments in an idempotent way


# Initial download
Download everything

In [3]:
# for day in get_market_dates(START_DATE, END_DATE):
#     destination = DATA_PATH + f'raw/flatfiles/{day.isoformat()}.csv.gz'
#     s3.download_file('flatfiles', 
#                 f'us_stocks_sip/minute_aggs_v1/{day.year}/{day.strftime("%m")}/{day.isoformat()}.csv.gz', 
#                 destination)

In [ ]:
df = pd.read_csv(DATA_PATH + f"raw/flatfiles/2016-06-06.csv.gz")

In [5]:
df

,ticker,volume,open,close,high,low,window_start,transactions
0,A,16484,45.72,45.780,45.78,45.72,1465219920000000000,35
1,A,1128,45.82,45.840,45.84,45.81,1465219980000000000,22
2,A,2637,45.81,45.845,45.88,45.81,1465220040000000000,31
3,A,953,45.84,45.800,45.84,45.80,1465220100000000000,18
4,A,4591,45.80,45.760,45.81,45.75,1465220160000000000,46
...,...,...,...,...,...,...,...,...
1263825,ZYNE,466,9.67,9.680,9.68,9.67,1465242840000000000,8
1263826,ZYNE,693,9.67,9.685,9.69,9.67,1465242960000000000,10
1263827,ZYNE,500,9.68,9.690,9.69,9.68,1465243020000000000,7
1263828,ZYNE,296,9.69,9.690,9.69,9.69,1465243080000000000,10


Extract and concatenate into monthly files

In [5]:
def concatenate_and_save(start_date, end_date, file_name):
    # Get a list of what we already have
    files = []
    # available_files = os.listdir(DATA_PATH + 'raw/flatfiles')
    # available_dates = [date.fromisoformat(file.replace(".csv.gz", "")) for file in available_files if file.endswith(".csv.gz")]
    
    market_dates = get_market_dates(start_date, end_date)
    if len(market_dates) == 0:
        return
    for day in market_dates:
        # if day >= start_date and day <= end_date and day in available_dates:
        destination = DATA_PATH + f'raw/flatfiles/{day.isoformat()}.csv.gz'
        with gzip.open(destination) as f:
            all_bars = pd.read_csv(f)
            all_bars = all_bars[['window_start', 'ticker', 'open', 'high', 'low', 'close', 'volume']]
            all_bars = all_bars.rename(columns={'window_start': 'datetime'})
            all_bars = all_bars.set_index('datetime')
            all_bars.index = pd.to_datetime(all_bars.index, unit='ns') # Convert to datetime (UTC-naive)
            # Make UTC aware (in order to convert)
            # Convert UTC to ET
            # Make timezone naive
            all_bars.index = all_bars.index.tz_localize(pytz.UTC).tz_convert("US/Eastern").tz_localize(None)  

            files.append(all_bars)
            print(day)
        
    all_bars = pd.concat(files)
    all_bars = all_bars.reset_index()
    all_bars = all_bars.set_index('ticker')
    all_bars = all_bars.sort_index()

    all_bars.to_parquet(DATA_PATH + f"raw/flatfiles/{file_name}.parquet", engine="fastparquet", compression="snappy", row_group_offsets=25000)

In [6]:
for year in range(2016, 2026 + 1):
    files = []
    for month in range(1, 12 + 1):
        _, end_date = calendar.monthrange(year, month)
        concatenate_and_save(date(year, month, 1), date(year, month, end_date), f"{year}-{month}")

2016-06-06
2016-06-07
2016-06-08
2016-06-09
2016-06-10
2016-06-13
2016-06-14
2016-06-15
2016-06-16
2016-06-17
2016-06-20
2016-06-21
2016-06-22
2016-06-23
2016-06-24
2016-06-27
2016-06-28
2016-06-29
2016-06-30
2016-07-01
2016-07-05
2016-07-06
2016-07-07
2016-07-08
2016-07-11
2016-07-12
2016-07-13
2016-07-14
2016-07-15
2016-07-18
2016-07-19
2016-07-20
2016-07-21
2016-07-22
2016-07-25
2016-07-26
2016-07-27
2016-07-28
2016-07-29
2016-08-01
2016-08-02
2016-08-03
2016-08-04
2016-08-05
2016-08-08
2016-08-09
2016-08-10
2016-08-11
2016-08-12
2016-08-15
2016-08-16
2016-08-17
2016-08-18
2016-08-19
2016-08-22
2016-08-23
2016-08-24
2016-08-25
2016-08-26
2016-08-29
2016-08-30
2016-08-31
2016-09-01
2016-09-02
2016-09-06
2016-09-07
2016-09-08
2016-09-09
2016-09-12
2016-09-13
2016-09-14
2016-09-15
2016-09-16
2016-09-19
2016-09-20
2016-09-21
2016-09-22
2016-09-23
2016-09-26
2016-09-27
2016-09-28
2016-09-29
2016-09-30
2016-10-03
2016-10-04
2016-10-05
2016-10-06
2016-10-07
2016-10-10
2016-10-11
2016-10-12

Split the monthly files which contains all ticker into individual ticker files.

In [ ]:
for year in range(2016, 2026+1):
    files = []
    for month in range(1, 12+1):
        print(f'{datetime.now()} | {year}-{month}')
        if not os.path.isfile(DATA_PATH + f"raw/flatfiles/monthly/{year}-{month}.parquet"):
            continue
        
        all_bars = pd.read_parquet(DATA_PATH + f"raw/flatfiles/monthly/{year}-{month}.parquet")
        all_bars = all_bars[~all_bars.index.isna()]

        if all_bars['datetime'].min().year != year or all_bars['datetime'].min().month != month:
            raise Exception(f'{year} | {month} HAS OUT OF BOUND DATES')

        # File names are case insensitive! This lead to big data errors (e.g. TpC and TPC are merged)
        # So we simply remove all tickers that have small letters.
        # Which we don't need anyways, because small letter = non-common stock.
        for ticker in list(filter(lambda ticker: not any(s.islower() for s in ticker), list(all_bars.index.unique()))):
            bars = all_bars.loc[ticker]
            if isinstance(bars, pd.Series):
                bars = all_bars.loc[[ticker]]
            bars = bars[['datetime', 'open', 'high', 'low', 'close', 'volume']]
            bars = bars.set_index('datetime').sort_index()
            
            # Windows quirk note: you cannot save files called 'prn'. Of course there is a ticker that is named PRN...
            # So we name it 'PRN_'. I hope there are no tickers named NULL or something please.
            # DEV: NO NEED TO DO THIS FOR MAC.
            # if ticker == 'PRN':
            #     ticker = 'PRN_'

            if os.path.isfile(DATA_PATH + f'raw/m1/{ticker}.parquet'):
                write(DATA_PATH + f"raw/m1/{ticker}.parquet", bars, append=True, compression="snappy", row_group_offsets=25000)
            else:
                bars.to_parquet(DATA_PATH + f"raw/m1/{ticker}.parquet", engine="fastparquet", compression="snappy", row_group_offsets=25000)

2026-06-06 15:44:06.199880 | 2026-3
2026-06-06 15:44:39.978248 | 2026-4
2026-06-06 15:44:39.978638 | 2026-5
2026-06-06 15:44:39.978669 | 2026-6
2026-06-06 15:44:39.978684 | 2026-7
2026-06-06 15:44:39.978694 | 2026-8
2026-06-06 15:44:39.978703 | 2026-9
2026-06-06 15:44:39.978715 | 2026-10
2026-06-06 15:44:39.978724 | 2026-11
2026-06-06 15:44:39.978733 | 2026-12


In [ ]:
""" for year in range(2003, 2024+1):
    files = []
    for month in range(1, 12+1):
        print(f'{datetime.now()} | {year}-{month}')
        if not os.path.isfile(DATA_PATH + f"raw/flatfiles/{year}-{month}.parquet"):
            continue
        
        all_bars = pd.read_parquet(DATA_PATH + f"raw/flatfiles/{year}-{month}.parquet")
        all_bars = all_bars[~all_bars.index.isna()]

        if all_bars['datetime'].min().year != year or all_bars['datetime'].min().month != month:
            raise Exception(f'{year} | {month} HAS OUT OF BOUND DATES')

        # File names are case insensitive! This lead to big data errors (e.g. TpC and TPC are merged)
        # So we simply remove all tickers that have small letters.
        # Which we don't need anyways, because small letter = non-common stock.
        for ticker in list(filter(lambda ticker: not any(s.islower() for s in ticker), list(all_bars.index.unique()))):
            bars = all_bars.loc[ticker]
            if isinstance(bars, pd.Series):
                bars = all_bars.loc[[ticker]]
            bars = bars[['datetime', 'open', 'high', 'low', 'close', 'volume']]
            bars = bars.set_index('datetime').sort_index()
            
            # Windows quirk note: you cannot save files called 'prn'. Of course there is a ticker that is named PRN...
            # So we name it 'PRN_'. I hope there are no tickers named NULL or something please.
            if ticker == 'PRN':
                ticker = 'PRN_'

            if os.path.isfile(DATA_PATH + f'raw/m1/{ticker}.parquet'):
                write(DATA_PATH + f"raw/m1/{ticker}.parquet", bars, append=True, compression="snappy", row_group_offsets=25000)
            else:
                bars.to_parquet(DATA_PATH + f"raw/m1/{ticker}.parquet", engine="fastparquet", compression="snappy", row_group_offsets=25000) """

The old version of the above code runs very fast, up to end 2022. Then it becomes 20-50x slower. I found the problem:

Pandas dataframe lookup uses hash-tables so the time complexity is O(1). However, if you have null values in the index this does not apply anymore! The bars from end 2022 and upwards have null values.

# Updates
Process day-by-day

In [ ]:
def process_flatfile(local_file_path):
    """Unzips the flat file and split or append it to ticker files.
    """
    with gzip.open(local_file_path) as f:
        all_bars = pd.read_csv(f)
        all_bars = all_bars[['window_start', 'ticker', 'open', 'high', 'low', 'close', 'volume']]
        all_bars = all_bars.rename(columns={'window_start': 'datetime'})
        all_bars = all_bars.set_index('datetime')
        all_bars.index = pd.to_datetime(all_bars.index, unit='ns') # Convert to datetime (UTC-naive)
        all_bars.index = all_bars.index.tz_localize(pytz.UTC)  # Make UTC aware (in order to convert)
        all_bars.index = all_bars.index.tz_convert("US/Eastern")  # Convert UTC to ET
        all_bars.index = all_bars.index.tz_localize(None)  # Make timezone naive
        
        for ticker in all_bars['ticker'].unique():
            bars = all_bars[all_bars['ticker'] == ticker]
            bars = bars[['open', 'high', 'low', 'close', 'volume']]

            if os.path.isfile(DATA_PATH + f'raw/m1/{ticker}.parquet'):
                write(DATA_PATH + f"raw/m1/{ticker}.parquet", bars, append=True, compression="snappy", row_group_offsets=25000)
            else:
                bars.to_parquet(DATA_PATH + f"raw/m1/{ticker}.parquet", engine="fastparquet", compression="snappy", row_group_offsets=25000)

In [ ]:
for day in get_market_dates(START_DATE, END_DATE):
    destination = DATA_PATH + f'raw/{day.isoformat()}.csv.gz'
    s3.download_file('flatfiles', 
                 f'us_stocks_sip/minute_aggs_v1/{day.year}/{day.strftime("%m")}/{day.isoformat()}.csv.gz', 
                 destination)
    process_flatfile(destination)